# 🚜 FarmifAI: Evaluación End-to-End del Sistema RAG Completo
### Pipeline Integrado: Recuperador Híbrido (BM25 + Dense Multilingual-E5) + Reranker (Cross-Encoder) + Generador SLM (LLaMA.cpp)

---

## 🎯 Objetivo de este Notebook
A diferencia de la evaluación aislada del modelo generador (donde el contexto era estático y perfecto), este notebook evalúa el **sistema RAG completo de extremo a extremo (End-to-End)** bajo condiciones reales de producción:
1. **Recuperación Dinámica**: A partir de la pregunta del agricultor en el dataset, el sistema consulta en tiempo real la base de conocimientos (`knowledge_base.json`) mediante búsqueda híbrida léxico-semántica y reordenamiento con Cross-Encoder.
2. **Inyección Dinámica de Contexto**: Los fragmentos recuperados en el Top-K final se formatean e inyectan dentro de la etiqueta `<knowledge>` del prompt.
3. **Generación con SLM Local**: El modelo SLM cuantizado en formato GGUF infiere la respuesta con etiquetas `<reasoning>` y `<answer>`.
4. **Métricas de Evaluación Holística**:
   - **Tríada RAG (LLM as a Judge)**:
     - *Context Relevancy*: ¿El contexto recuperado por el RAG es relevante y libre de ruido?
     - *Faithfulness*: ¿La respuesta generada está fundamentada en el contexto **recuperado** dinámicamente?
     - *Answer Relevancy*: ¿La respuesta generada responde directamente a la inquietud del agricultor?
   - **Calidad Textual y Lingüística (G-Eval)**: Coherencia, Consistencia y Fluidez evaluadas con cadena de razonamiento (Chain of Thought).
   - **Similitud Semántica Factual (BERTScore)**: Precision, Recall y F1 frente a la respuesta de referencia experta.
   - **Eficiencia y Latencia (Profiling de Sistema)**: Tiempos de búsqueda (BM25, Semántica, Reranker), tiempo de inferencia del SLM, latencia total extremo a extremo (E2E) y velocidad de generación (tokens/segundo).


## 1. Setup e Instalación de Dependencias

Instalamos únicamente las dependencias requeridas para el sistema RAG híbrido, inferencia local con `llama-cpp-python` acelerado por GPU CUDA, `bert-score`, y clientes de LLM as a Judge.


In [ ]:
%%capture
import os, sys, importlib.util

print("[INFO] Actualizando gestor de paquetes uv...")
!pip install --upgrade -qqq uv

print("[INFO] Instalando librerías de evaluación, RAG híbrido y utilidades...")
!uv pip install -qqq huggingface_hub sentence-transformers rank-bm25 bert-score openai pandas matplotlib seaborn tqdm nltk torch

import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# --- INSTALACIÓN OPTIMIZADA DE LLAMA-CPP-PYTHON PARA GPU EN COLAB ---
# 1. En Google Colab con GPU (CUDA 12.x), el wheel precompilado cu121 es binariamente compatible
print("[INFO] Instalando llama-cpp-python precompilado para GPU CUDA (cu121)...")
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 --upgrade --no-cache-dir

# 2. Verificación estricta de soporte CUDA
import llama_cpp
gpu_supported = llama_cpp.llama_supports_gpu_offload()

if not gpu_supported:
    print("[WARNING] El wheel precompilado no activó soporte CUDA. Compilando con GGML_CUDA=on (máx 2-3 min)...")
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=all-major"
    os.environ["MAX_JOBS"] = "4"
    !pip install --no-cache-dir -q llama-cpp-python --force-reinstall
    import importlib
    importlib.reload(llama_cpp)
    gpu_supported = llama_cpp.llama_supports_gpu_offload()

print(f"[OK] Instalación de dependencias completada exitosamente.")
print(f"✅ llama-cpp versión: {llama_cpp.__version__}")
print(f"🚀 Soporte GPU CUDA activo: {gpu_supported}")

In [ ]:
import torch
import psutil

print("=" * 60)
print(" 🖥️ DIAGNÓSTICO DEL ENTORNO DE EVALUACIÓN END-TO-END")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detectada: {gpu_name}")
    print(f"📊 VRAM Disponible: {vram_total:.2f} GB")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU CUDA. La inferencia local y reranking serán lentos en CPU.")
    print("Por favor, ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' -> Selecciona 'T4 GPU'.")

ram_total = psutil.virtual_memory().total / (1024**3)
print(f"💾 RAM del Sistema: {ram_total:.2f} GB")
print("=" * 60)

## 2. Selección y Descarga Dinámica del Modelo GGUF (SLM)

Nos conectamos a Hugging Face Hub para inspeccionar y descargar el modelo conversacional cuantizado en formato GGUF (`FarmifAI/FarmifAI_1.3_GGUF`).


In [ ]:
from huggingface_hub import HfApi, hf_hub_download

HF_REPO_ID = "FarmifAI/FarmifAI_1.3_GGUF"
DEFAULT_GGUF_FILE = "FarmifAI_1.3.F16.gguf"

api = HfApi()
try:
    print(
        f"[INFO] Listando modelos disponibles en el repositorio '{HF_REPO_ID}'...")
    repo_files = api.list_repo_files(repo_id=HF_REPO_ID)
    gguf_files = sorted([f for f in repo_files if f.endswith(".gguf")])

    print("\n📦 Cuantizaciones GGUF disponibles:")
    for idx, filename in enumerate(gguf_files):
        marker = " 👈 (Por defecto)" if filename == DEFAULT_GGUF_FILE else ""
        print(f"  [{idx}] {filename}{marker}")
except Exception as e:
    print(
        f"[WARNING] No se pudo obtener la lista remota ({e}). Usando lista por defecto.")
    gguf_files = [DEFAULT_GGUF_FILE]

# --- SELECCIÓN DEL MODELO ---
SELECTED_FILE = DEFAULT_GGUF_FILE

if SELECTED_FILE not in gguf_files and len(gguf_files) > 0:
    SELECTED_FILE = gguf_files[0]

print(f"\n[INFO] Descargando modelo seleccionado: {SELECTED_FILE}...")
model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=SELECTED_FILE)
print(f"✅ Modelo descargado y listo en ruta local: {model_path}")

## 3. Carga del Corpus de Conocimiento (`knowledge_base.json`) y Modelos RAG

Cargamos la base de conocimientos agrícolas, preprocesamos los textos con Snowball Stemmer para el índice BM25, generamos/cargamos los embeddings densos `Multilingual-E5` y cargamos el Cross-Encoder para el reordenamiento fino.


In [ ]:
import json
import os
import pickle
import re
import time
import unicodedata
from typing import Any
import numpy as np
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("✅ Ejecutando en entorno Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Ejecutando en entorno local")

CHUNKS_FILENAME = "knowledge_base.json"
EMBEDDINGS_FILENAME = "embeddings_chunks_e5_small.pkl"
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"
RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

# --- 1. CARGA DE CHUNKS ---


def load_chunks(filename: str) -> list[dict]:
    if not os.path.exists(filename):
        if IN_COLAB:
            print(f"📂 Sube el archivo '{filename}':")
            uploaded = colab_files.upload()
            if filename not in uploaded:
                raise FileNotFoundError(f"No se subió '{filename}'.")
        else:
            raise FileNotFoundError(f"No se encontró '{filename}'.")
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["chunks"]


chunks = load_chunks(CHUNKS_FILENAME)
print(f"✅ {len(chunks)} chunks cargados exitosamente desde '{CHUNKS_FILENAME}'.")

# --- 2. PREPROCESAMIENTO PARA BM25 (ESPAÑOL) ---
stemmer = SnowballStemmer("spanish")


def remove_accents(text: str) -> str:
    text = unicodedata.normalize("NFD", text)
    return "".join(c for c in text if unicodedata.category(c) != "Mn")


spanish_stopwords = set(remove_accents(w.lower())
                        for w in stopwords.words("spanish"))


def preprocess_spanish(text: str) -> list[str]:
    text = text.lower()
    text = remove_accents(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [t for t in text.split() if len(
        t) > 1 and t not in spanish_stopwords]
    stemmed_tokens = [stemmer.stem(t) for t in tokens]
    return [t for t in stemmed_tokens if len(t) > 1]


print("⏳ Preprocesando textos para índice léxico BM25...")
t0 = time.time()
corpus_tokens = [preprocess_spanish(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"✅ Índice BM25 construido en {time.time() - t0:.2f}s.")

# --- 3. EMBEDDINGS DENSOS (MULTILINGUAL-E5) ---


def load_or_generate_embeddings(chunks: list[dict], model_name: str, embeddings_file: str):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(
        f"⏳ Cargando modelo de embeddings: {model_name} en {device.upper()}...")
    model = SentenceTransformer(model_name, device=device)

    embeddings = None
    if os.path.exists(embeddings_file):
        print(f"📦 Archivo de embeddings encontrado: '{embeddings_file}'")
        with open(embeddings_file, "rb") as f:
            embeddings = pickle.load(f)
        print(f"✅ Embeddings cargados desde disco: shape={embeddings.shape}")
    elif IN_COLAB:
        print(
            f"📂 ¿Tienes embeddings precalculados? Sube '{embeddings_file}' o presiona 'Cancel' para generarlos:")
        try:
            uploaded = colab_files.upload()
            if embeddings_file in uploaded:
                with open(embeddings_file, "rb") as f:
                    embeddings = pickle.load(f)
                print(
                    f"✅ Embeddings cargados desde subida: shape={embeddings.shape}")
        except Exception:
            print("ℹ️ No se subieron embeddings precalculados. Generando desde cero...")

    if embeddings is not None and (embeddings.shape[0] != len(chunks) or embeddings.shape[1] != 384):
        print(
            f"⚠️ Dimensión incompatible ({embeddings.shape}). Regenerando...")
        embeddings = None

    if embeddings is None:
        texts_with_prefix = [f"passage: {chunk['text']}" for chunk in chunks]
        print(
            f"⏳ Generando embeddings para {len(texts_with_prefix)} chunks...")
        t0 = time.time()
        embeddings = model.encode(
            texts_with_prefix,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        print(
            f"✅ Embeddings generados en {time.time() - t0:.1f}s | shape={embeddings.shape}")
        with open(embeddings_file, "wb") as f:
            pickle.dump(embeddings, f)
        print(f"💾 Embeddings guardados en '{embeddings_file}'")

    return embeddings, model


chunk_embeddings, embedding_model = load_or_generate_embeddings(
    chunks, EMBEDDING_MODEL_NAME, EMBEDDINGS_FILENAME
)

In [ ]:
class HybridRAGPipeline:
    """Pipeline RAG Híbrido con medición de latencias por etapa (BM25 + E5 + RRF + Cross-Encoder)."""

    def __init__(
        self,
        chunks: list[dict],
        bm25_index: BM25Okapi,
        corpus_tokens: list[list[str]],
        chunk_embeddings: np.ndarray,
        embedding_model: SentenceTransformer,
        reranker_model_name: str = RERANKER_MODEL_NAME,
    ):
        self.chunks = chunks
        self.bm25 = bm25_index
        self.corpus_tokens = corpus_tokens
        self.chunk_embeddings = chunk_embeddings
        self.embedding_model = embedding_model

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(
            f"⏳ Cargando modelo Cross-Encoder Reranker ({reranker_model_name}) en {device.upper()}...")
        self.reranker = CrossEncoder(reranker_model_name, device=device)
        print(f"✅ Reranker listo.")

    def search_bm25(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        query_tokens = preprocess_spanish(query)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

    def search_semantic(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        query_with_prefix = f"query: {query}"
        query_embedding = self.embedding_model.encode(
            query_with_prefix, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
        )
        similarities = self.chunk_embeddings @ query_embedding
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(int(i), float(similarities[i])) for i in top_indices]

    @staticmethod
    def reciprocal_rank_fusion(
        results_lists: list[list[tuple[int, float]]],
        k: int = 60,
    ) -> list[tuple[int, float]]:
        rrf_scores: dict[int, float] = {}
        for results in results_lists:
            for rank, (idx, _) in enumerate(results):
                rrf_scores[idx] = rrf_scores.get(
                    idx, 0.0) + 1.0 / (k + rank + 1)
        sorted_results = sorted(
            rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results

    def rerank(self, query: str, candidate_indices: list[int]) -> list[tuple[int, float]]:
        if not candidate_indices:
            return []
        pairs = [(query, self.chunks[i]["text"]) for i in candidate_indices]
        scores = self.reranker.predict(pairs, show_progress_bar=False)
        scored = list(zip(candidate_indices, [float(s) for s in scores]))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def query(
        self,
        user_query: str,
        top_k_lexical: int = 10,
        top_k_semantic: int = 10,
        top_k_final: int = 3
    ) -> dict[str, Any]:
        """Ejecuta la recuperación híbrida y perfila el tiempo de cada fase."""
        # 1. Búsqueda Léxica
        t0 = time.perf_counter()
        bm25_results = self.search_bm25(user_query, top_k=top_k_lexical)
        t_bm25 = time.perf_counter() - t0

        # 2. Búsqueda Semántica
        t0 = time.perf_counter()
        semantic_results = self.search_semantic(
            user_query, top_k=top_k_semantic)
        t_semantic = time.perf_counter() - t0

        # 3. Fusión RRF
        t0 = time.perf_counter()
        hybrid_results = self.reciprocal_rank_fusion(
            [bm25_results, semantic_results])
        t_fusion = time.perf_counter() - t0

        # 4. Reranking con Cross-Encoder
        candidate_indices = [idx for idx, _ in hybrid_results]
        t0 = time.perf_counter()
        reranked_results = self.rerank(user_query, candidate_indices)
        t_rerank = time.perf_counter() - t0

        t_retrieval = t_bm25 + t_semantic + t_fusion + t_rerank

        final_chunks = []
        for idx, rerank_score in reranked_results[:top_k_final]:
            final_chunks.append({
                **self.chunks[idx],
                "rerank_score": float(rerank_score),
                "_index": int(idx)
            })

        return {
            "chunks": final_chunks,
            "timings": {
                "time_bm25": t_bm25,
                "time_semantic": t_semantic,
                "time_fusion": t_fusion,
                "time_rerank": t_rerank,
                "time_retrieval": t_retrieval
            }
        }


# Instanciar el pipeline RAG
rag_pipeline = HybridRAGPipeline(
    chunks=chunks,
    bm25_index=bm25,
    corpus_tokens=corpus_tokens,
    chunk_embeddings=chunk_embeddings,
    embedding_model=embedding_model,
    reranker_model_name=RERANKER_MODEL_NAME
)
print("✅ Pipeline RAG Híbrido inicializado y listo para consultas.")

## 4. Carga y Preparación del Dataset de Evaluación (`dataset_agricola_eval.jsonl`)

Cargamos el dataset de preguntas agrícolas y respuestas de referencia. Nótese que a diferencia de la evaluación previa, **la etiqueta `<knowledge>` del dataset NO será entregada al modelo**, sino que la obtendremos dinámicamente llamando a `rag_pipeline.query(question)`.


In [ ]:
target_dataset_filename = "dataset_agricola_eval.jsonl"

if os.path.exists(target_dataset_filename):
    print(
        f"[INFO] El archivo '{target_dataset_filename}' ya existe localmente. Omitiendo la carga.")
    dataset_filename = target_dataset_filename
else:
    print(f"[INFO] Selecciona y sube tu archivo '{target_dataset_filename}':")
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError(
            "❌ No se subió ningún archivo. Por favor ejecuta la celda nuevamente.")
    dataset_filename = list(uploaded.keys())[0]
    print(f"✅ Archivo cargado: {dataset_filename}")

raw_data = []
with open(dataset_filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                raw_data.append(json.loads(line))
            except Exception as e:
                print(f"[WARNING] Línea ignorada por error JSON: {e}")

print(f"[INFO] Total de muestras encontradas en el dataset: {len(raw_data)}")


def parse_eval_sample(sample, index):
    messages = sample.get("messages", [])
    user_msg = next((m["content"]
                    for m in messages if m["role"] == "user"), "")
    assistant_msg = next((m["content"]
                         for m in messages if m["role"] == "assistant"), "")

    # Extraer la pregunta limpia (removiendo cualquier etiqueta <knowledge> que viniera en el dataset)
    question = re.sub(r'<knowledge>.*?</knowledge>', '',
                      user_msg, flags=re.DOTALL).strip()
    if not question:
        question = user_msg.strip()

    # Extraer la respuesta de referencia (el contenido de <answer> para evaluación comparativa)
    ans_match = re.search(r'<answer>(.*?)</answer>', assistant_msg, re.DOTALL)
    ref_answer = ans_match.group(1).strip(
    ) if ans_match else assistant_msg.strip()

    return {
        "sample_index": index,
        "question": question,
        "reference_full": assistant_msg,
        "reference_answer": ref_answer
    }


eval_dataset = [parse_eval_sample(sample, idx)
                for idx, sample in enumerate(raw_data)]
print(
    f"✅ Dataset estructurado correctamente ({len(eval_dataset)} preguntas para evaluar en RAG).")

if eval_dataset:
    print("\n🔍 Ejemplo de Muestra [Índice 0]:")
    print(f"  Pregunta: {eval_dataset[0]['question'][:120]}...")
    print(
        f"  Respuesta de Referencia: {eval_dataset[0]['reference_answer'][:120]}...")

## 5. Configuración de Motores de Inferencia (SLM Local y LLM as a Judge)

- **SLM Local**: Cargado con `llama-cpp-python` optimizado para GPU.
- **LLM as a Judge**: Cliente compatible con OpenAI API para llamar a **DeepSeek** (`deepseek-chat`) u **OpenRouter** (`nvidia/nemotron-3-ultra-550b-a55b:free`).


In [ ]:
from llama_cpp import Llama
import llama_cpp
from openai import OpenAI
import getpass

# --- 1. INICIALIZAR LLAMA.CPP LOCAL CON ACELERACIÓN GPU ---
print(f"[INFO] Cargando modelo local en Llama.cpp: {SELECTED_FILE}...")
assert llama_cpp.llama_supports_gpu_offload(
), "⚠️ ADVERTENCIA CRÍTICA: llama-cpp-python está en modo CPU. Verifica la celda 2."

llm_local = Llama(
    model_path=model_path,
    n_ctx=4096,          # Ventana de contexto para soportar chunks RAG
    n_batch=512,         # Tamaño de batch para procesamiento acelerado de prompts en GPU
    n_gpu_layers=-1,     # Offload de todas las capas a GPU CUDA
    seed=42,
    verbose=False
)
print(
    f"✅ Motor local Llama.cpp inicializado exitosamente en GPU (GPU Offload: {llama_cpp.llama_supports_gpu_offload()}).")

# --- 2. CONFIGURACIÓN DEL LLM AS A JUDGE ---
# Opciones de PROVEEDOR: "deepseek" o "openrouter"
JUDGE_PROVIDER = "deepseek"   # Cambiar a "openrouter" según tu preferencia

# 🔑 Ingresa tu API Key (si está vacía, se te solicitará interactivamente)
JUDGE_API_KEY = ""

JUDGE_MODEL = "deepseek-chat" if JUDGE_PROVIDER == "deepseek" else "nvidia/nemotron-3-ultra-550b-a55b:free"

if not JUDGE_API_KEY:
    print(f"\n🔑 Por favor ingresa tu API Key para {JUDGE_PROVIDER.upper()}:")
    JUDGE_API_KEY = getpass.getpass()


def init_judge_client(provider, api_key):
    if provider.lower() == "deepseek":
        return OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
    elif provider.lower() == "openrouter":
        return OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1",
            default_headers={
                "HTTP-Referer": "https://github.com/FarmifAI",
                "X-Title": "FarmifAI RAG System Evaluation"
            }
        )
    else:
        raise ValueError(f"Proveedor no soportado: {provider}")


judge_client = init_judge_client(JUDGE_PROVIDER, JUDGE_API_KEY)
print(
    f"✅ Cliente LLM Judge configurado ({JUDGE_PROVIDER.upper()} - Modelo: {JUDGE_MODEL}).")

## 6. Implementación de las Métricas de Evaluación

En este sistema RAG integral evaluamos:
1. **BERTScore**: Precision, Recall y F1 frente a la respuesta de referencia experta.
2. **Tríada RAG (LLM Judge)**:
   - **`Context Relevancy`**: ¿El contexto recuperado dinámicamente es pertinente y libre de ruido?
   - **`Faithfulness`**: ¿La respuesta generada se respalda fielmente en el contexto **recuperado por el RAG**?
   - **`Answer Relevancy`**: ¿La respuesta es directa y completa respecto a la pregunta del agricultor?
3. **G-Eval (LLM Judge con CoT)**: Coherencia, Consistencia y Fluidez lingüística (escala 1 a 5).
4. **Métricas de Latencia y Eficiencia**: Tiempos por componente ($t_{\text{bm25}}$, $t_{\text{semantic}}$, $t_{\text{rerank}}$, $t_{\text{retrieval}}$, $t_{\text{generation}}$, $t_{\text{e2e}}$) y velocidad de generación (tokens/segundo).


In [ ]:
import random
import bert_score
from bert_score import BERTScorer

print("[INFO] Inicializando evaluador BERTScore multilingüe (español)...")
bert_scorer = BERTScorer(
    lang='es',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    rescale_with_baseline=False
)
print("✅ BERTScore inicializado exitosamente.")


def extract_generated_answer(text: str) -> str:
    """Extrae el contenido de la etiqueta <answer>, o hace fallback al texto limpio."""
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    clean = re.sub(r'<reasoning>.*?</reasoning>',
                   '', text, flags=re.DOTALL).strip()
    return clean.replace('<answer>', '').replace('</answer>', '').strip()

# --- MÉTRICA: BERTSCORE ---


def eval_bertscore(ref_answer: str, gen_answer: str) -> dict:
    if not gen_answer or not ref_answer:
        return {"bertscore_precision": 0.0, "bertscore_recall": 0.0, "bertscore_f1": 0.0}
    P, R, F1 = bert_scorer.score([gen_answer], [ref_answer])
    return {
        "bertscore_precision": float(P[0].item()),
        "bertscore_recall": float(R[0].item()),
        "bertscore_f1": float(F1[0].item())
    }


# --- LLM AS A JUDGE: INVOCACIÓN ROBUSTA Y DEFENSIVA ---


def extract_json_from_response(content: str):
    """Extrae y parsea un objeto JSON de una cadena, tolerando markdown."""
    if not content:
        return None
    content_clean = content.strip()
    if "```" in content_clean:
        match = re.search(
            r"```(?:json)?\s*(\{.*?\})\s*```", content_clean, re.DOTALL)
        if match:
            content_clean = match.group(1).strip()
    if not (content_clean.startswith("{") and content_clean.endswith("}")):
        match = re.search(r"(\{.*\})", content_clean, re.DOTALL)
        if match:
            content_clean = match.group(1).strip()
    try:
        return json.loads(content_clean)
    except json.JSONDecodeError:
        return None


def call_llm_judge(system_prompt: str, user_prompt: str, max_retries: int = 5, base_delay: float = 2.0, max_delay: float = 30.0):
    """
    Ejecuta una llamada al LLM Judge con validación defensiva ante anomalías de OpenRouter
    (como 'choices: null' o rate limits) y reintentos con backoff exponencial y jitter.
    """
    for attempt in range(max_retries):
        try:
            response = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )

            # Validación defensiva ante respuestas truncadas, nulas o con 'choices: null'
            if not response or not hasattr(response, "choices") or response.choices is None or len(response.choices) == 0:
                err_msg = getattr(response, "error",
                                  None) or "response.choices es None o vacía"
                raise ValueError(f"API retornó respuesta inválida ({err_msg})")

            choice = response.choices[0]
            if not hasattr(choice, "message") or choice.message is None:
                raise ValueError("Respuesta de la API no contiene 'message'")

            content = choice.message.content
            if not content:
                raise ValueError(
                    "Contenido devuelto por el LLM Judge está vacío (None)")

            parsed = extract_json_from_response(content)
            if not parsed:
                raise ValueError(
                    f"No se pudo decodificar JSON válido del texto: {content[:100]}...")

            return parsed

        except Exception as e:
            delay = min(base_delay * (2 ** attempt) +
                        random.uniform(0.5, 1.5), max_delay)
            if attempt < max_retries - 1:
                print(
                    f"  [RETRY {attempt+1}/{max_retries}] Error en LLM Judge ({type(e).__name__}: {e}). Reintentando en {delay:.1f}s...")
                time.sleep(delay)
            else:
                print(
                    f"[ERROR] Falló llamada a LLM Judge tras {max_retries} intentos: {e}")
                return None


# --- LLM JUDGE: TRÍADA RAG (CONTEXT RELEVANCY + FAITHFULNESS + ANSWER RELEVANCY) ---
def eval_rag_triad(question: str, retrieved_knowledge: str, gen_answer: str, max_retries: int = 5) -> dict:
    """
    Evalúa la Tríada RAG. Si la API falla tras reintentos, asigna None para no sesgar promedios con ceros.
    """
    if not gen_answer:
        return {"context_relevancy": 0.0, "faithfulness": 0.0, "answer_relevancy": 0.0, "reasoning": "Respuesta vacía", "status": "EMPTY_ANSWER"}

    prompt_sys = (
        "Eres un evaluador experto y riguroso en sistemas RAG agrícolas. Evalúa la consulta, el contexto recuperado y la respuesta en 3 dimensiones clave (escala 0 a 5):\n"
        "1. 'context_relevancy': ¿Qué tan relevante, útil y libre de ruido es el contexto recuperado (<knowledge>) respecto a la pregunta del usuario? (5=altamente relevante y pertinente, 0=completamente irrelevante o puro ruido).\n"
        "2. 'faithfulness': ¿Qué tanto de lo afirmado en la respuesta generada está respaldado de manera fidedigna por el contexto recuperado (<knowledge>)? (5=100% respaldado sin invenciones/alucinaciones, 0=contradicción o alucinación total).\n"
        "3. 'answer_relevancy': ¿Qué tan directa, útil y completa es la respuesta a la pregunta del usuario? (5=excelente y directa, 0=evasiva o fuera de tema).\n\n"
        "Responde ESTRICTAMENTE en formato JSON con la siguiente estructura:\n"
        "{\"context_relevancy\": <int 0-5>, \"faithfulness\": <int 0-5>, \"answer_relevancy\": <int 0-5>, \"reasoning\": \"<breve justificación>\"}"
    )
    prompt_usr = (
        f"Pregunta del Agricultor: {question}\n\n"
        f"Contexto Recuperado Dinámicamente (<knowledge>):\n{retrieved_knowledge}\n\n"
        f"Respuesta Generada por el SLM:\n{gen_answer}"
    )

    res = call_llm_judge(prompt_sys, prompt_usr, max_retries=max_retries)
    if res and ("context_relevancy" in res or "faithfulness" in res or "answer_relevancy" in res):
        return {
            "context_relevancy": max(0.0, min(5.0, float(res.get("context_relevancy", 0.0)))),
            "faithfulness": max(0.0, min(5.0, float(res.get("faithfulness", 0.0)))),
            "answer_relevancy": max(0.0, min(5.0, float(res.get("answer_relevancy", 0.0)))),
            "reasoning": str(res.get("reasoning", "")),
            "status": "SUCCESS"
        }
    else:
        return {
            "context_relevancy": None,
            "faithfulness": None,
            "answer_relevancy": None,
            "reasoning": "Error de API tras reintentos",
            "status": "FAILED_API"
        }


# --- LLM JUDGE: G-EVAL (COHERENCIA, CONSISTENCIA, FLUIDEZ) ---
def eval_geval(question: str, gen_answer: str, max_retries: int = 5) -> dict:
    """
    Evalúa G-Eval (escala 1 a 5). Si la API falla tras reintentos, asigna None para proteger estadísticas.
    """
    if not gen_answer:
        return {"coherence": 0.0, "consistency": 0.0, "fluency": 0.0, "status": "EMPTY_ANSWER"}

    prompt_sys = (
        "Eres un juez evaluador riguroso utilizando el framework G-Eval. Evalúa la respuesta en 3 dimensiones en escala de 1 a 5:\n"
        "1. 'coherence': Flujo lógico y organización estructural de la respuesta (1=desorganizada, 5=perfectamente estructurada).\n"
        "2. 'consistency': Ausencia de contradicciones lógicas internas o con hechos agrícolas conocidos (1=contradictoria, 5=altamente consistente).\n"
        "3. 'fluency': Calidad gramatical en español, claridad y tono adaptado para un agricultor (1=incomprensible/robótico, 5=español natural, claro y excelente).\n\n"
        "Realiza un análisis paso a paso (CoT) y responde ESTRICTAMENTE en JSON:\n"
        "{\"coherence\": {\"score\": <int 1-5>, \"steps\": \"<análisis>\"}, \"consistency\": {\"score\": <int 1-5>, \"steps\": \"<análisis>\"}, \"fluency\": {\"score\": <int 1-5>, \"steps\": \"<análisis>\"}}"
    )
    prompt_usr = f"Pregunta: {question}\n\nRespuesta Generada:\n{gen_answer}"

    res = call_llm_judge(prompt_sys, prompt_usr, max_retries=max_retries)
    if res and ("coherence" in res or "consistency" in res or "fluency" in res):
        def _get_score(data, key):
            val = data.get(key)
            if isinstance(val, dict):
                return float(val.get("score", 0.0))
            elif isinstance(val, (int, float)):
                return float(val)
            return 0.0

        return {
            "coherence": max(1.0, min(5.0, _get_score(res, "coherence"))),
            "consistency": max(1.0, min(5.0, _get_score(res, "consistency"))),
            "fluency": max(1.0, min(5.0, _get_score(res, "fluency"))),
            "status": "SUCCESS"
        }
    else:
        return {
            "coherence": None,
            "consistency": None,
            "fluency": None,
            "status": "FAILED_API"
        }

## 7. Configuración de Checkpointing y Respaldo en Google Drive

El bucle guarda cada muestra evaluada de forma incremental en `rag_eval_checkpoint.jsonl` y lo sincroniza con Google Drive para garantizar que no se pierda progreso en caso de desconexión.


In [ ]:
import shutil

CHECKPOINT_FILE = "rag_eval_checkpoint.jsonl"
drive_dest_path = None

if not os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        pass

if IN_COLAB:
    try:
        from google.colab import drive
        print("[INFO] Conectando con Google Drive...")
        drive.mount('/content/drive')
        drive_folder = "/content/drive/MyDrive/FarmifAI_Evaluaciones"
        os.makedirs(drive_folder, exist_ok=True)
        drive_dest_path = os.path.join(drive_folder, "rag_eval_results.jsonl")
        print(f"✅ Respaldo configurado en Google Drive: {drive_dest_path}")
    except Exception as e:
        print(
            f"ℹ️ No se configuró respaldo en Google Drive ({e}). Los datos se guardarán localmente en '{CHECKPOINT_FILE}'.")

## 8. Bucle de Evaluación End-to-End con Checkpointing y Resiliencia

En cada paso:
1. Se consulta el pipeline RAG híbrido (`time_retrieval`, chunks con `rerank_score`).
2. Se ensambla el prompt ChatML con el contexto recuperado en `<knowledge>`.
3. Se genera la respuesta con el SLM local (`time_generation`, `tokens_per_second`).
4. Se calculan BERTScore, la Tríada RAG y G-Eval con el LLM Judge.
5. Se almacena el registro en el checkpoint.


In [ ]:
import concurrent.futures
from tqdm.notebook import tqdm

# 1. Recuperar progreso previo si la sesión fue interrumpida
completed_indices = set()
results_list = []

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    res_obj = json.loads(line)
                    completed_indices.add(res_obj["sample_index"])
                    results_list.append(res_obj)
                except Exception:
                    pass
    if completed_indices:
        print(
            f"🔄 Checkpoint encontrado: {len(completed_indices)} muestras ya evaluadas previamente.")

# 2. Configuración de tamaño de muestra
# Ajusta este número (por ej. 20 o 50 para una prueba rápida, o len(eval_dataset) para evaluación completa)
MAX_SAMPLES_TO_EVALUATE = 250  # len(eval_dataset)

print(
    f"\n🚀 INICIANDO EVALUACIÓN END-TO-END (Muestras objetivo: {min(MAX_SAMPLES_TO_EVALUATE, len(eval_dataset))})...")

for sample in tqdm(eval_dataset[:MAX_SAMPLES_TO_EVALUATE], desc="Evaluando Sistema RAG"):
    idx = sample["sample_index"]
    if idx in completed_indices:
        continue

    question = sample["question"]

    # --- PASO 1: RECUPERACIÓN RAG DINÁMICA ---
    rag_out = rag_pipeline.query(
        user_query=question,
        top_k_lexical=10,
        top_k_semantic=10,
        top_k_final=3
    )
    retrieved_chunks = rag_out["chunks"]
    timings = rag_out["timings"]

    # Construir texto formateado de <knowledge>
    knowledge_text = "\n\n".join([
        f"Documento: {c.get('document_id', 'Desconocido')} (Fragmento {c.get('chunk_number', 'N/A')})\nContenido: {c.get('text', '')}"
        for c in retrieved_chunks
    ]) if retrieved_chunks else "No hay contexto disponible."

    # --- PASO 2: CONSTRUIR PROMPT CHATML ---
    system_prompt = (
        "Eres un asistente inteligente. Analiza la información proporcionada en la etiqueta <knowledge> para responder a la solicitud del usuario. Responde estrictamente en el siguiente formato:\n"
        "<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>"
    )
    user_prompt = f"<knowledge>\n{knowledge_text}\n</knowledge>\n\n{question}"

    prompt_chatml = (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    # --- PASO 3: INFERENCIA DEL SLM LOCAL (LLAMA-CPP) ---
    t0_gen = time.perf_counter()
    gen_res = llm_local(
        prompt_chatml,
        max_tokens=512,
        stop=["<|im_end|>", "<|endoftext|>"],
        temperature=0.3,
        top_p=0.8
    )
    t_gen = time.perf_counter() - t0_gen

    generated_full = gen_res["choices"][0]["text"].strip()
    generated_answer = extract_generated_answer(generated_full)

    # Métricas de tokens
    usage = gen_res.get("usage", {})
    tokens_gen = usage.get("completion_tokens", len(generated_full.split()))
    tokens_prompt = usage.get("prompt_tokens", len(prompt_chatml.split()))
    tokens_per_sec = (tokens_gen / t_gen) if t_gen > 0 else 0.0
    t_e2e = timings["time_retrieval"] + t_gen

    # --- PASO 4: CÁLCULO DE MÉTRICAS DE CALIDAD ---
    bert_scores = eval_bertscore(sample["reference_answer"], generated_answer)
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        f_triad = executor.submit(
            eval_rag_triad, question, knowledge_text, generated_answer)
        f_geval = executor.submit(eval_geval, question, generated_answer)
        triad_res = f_triad.result()
        geval_res = f_geval.result()
    if triad_res.get("status") == "FAILED_API" or geval_res.get("status") == "FAILED_API":
        tqdm.write(
            f"  ⚠️ [AVISO] Muestra #{idx}: Falló API LLM Judge (Tríada: {triad_res.get('status')}, G-Eval: {geval_res.get('status')}). Contexto y respuesta a salvo; reevalúa en celda 8.1.")

    # --- PASO 5: CONSOLIDAR REGISTRO ---
    eval_record = {
        "sample_index": idx,
        "question": question,
        "reference_answer": sample["reference_answer"],
        "retrieved_chunks": [
            {
                "document_id": c.get("document_id"),
                "chunk_number": c.get("chunk_number"),
                "rerank_score": c.get("rerank_score"),
                "text": c.get("text")
            } for c in retrieved_chunks
        ],
        "retrieved_knowledge": knowledge_text,
        "generated_full": generated_full,
        "generated_answer": generated_answer,
        "metrics": {
            # Tríada RAG (LLM Judge 0 a 5)
            "context_relevancy": triad_res["context_relevancy"],
            "faithfulness": triad_res["faithfulness"],
            "answer_relevancy": triad_res["answer_relevancy"],
            "triad_reasoning": triad_res["reasoning"],
            "judge_triad_status": triad_res.get("status", "SUCCESS"),
            # G-Eval (LLM Judge 1 a 5)
            "geval_coherence": geval_res["coherence"],
            "geval_consistency": geval_res["consistency"],
            "geval_fluency": geval_res["fluency"],
            "judge_geval_status": geval_res.get("status", "SUCCESS"),
            # BERTScore (0 a 1)
            "bertscore_precision": bert_scores["bertscore_precision"],
            "bertscore_recall": bert_scores["bertscore_recall"],
            "bertscore_f1": bert_scores["bertscore_f1"],
            # Latencias (segundos) y Throughput (tok/s)
            "time_bm25": timings["time_bm25"],
            "time_semantic": timings["time_semantic"],
            "time_fusion": timings["time_fusion"],
            "time_rerank": timings["time_rerank"],
            "time_retrieval": timings["time_retrieval"],
            "time_generation": t_gen,
            "time_e2e": t_e2e,
            "tokens_generated": tokens_gen,
            "tokens_prompt": tokens_prompt,
            "tokens_per_second": tokens_per_sec
        }
    }

    # Guardar en checkpoint local
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(eval_record, ensure_ascii=False) + "\n")

    # Respaldo en Drive si está disponible
    if drive_dest_path:
        try:
            shutil.copy(CHECKPOINT_FILE, drive_dest_path)
        except Exception:
            pass

    completed_indices.add(idx)
    results_list.append(eval_record)

print(
    f"\n✅ ¡EVALUACIÓN COMPLETADA! Total de muestras evaluadas: {len(results_list)}.")

### 🔄 8.1 Reevaluación Rápida y Rescate de LLM Judge en RAG (In-Notebook)

Ejecuta la siguiente celda si durante la evaluación del sistema RAG alguna muestra tuvo fallo de API (`FAILED_API`, `None` o `0.0`).
- **Cero Cómputo Redundante:** Utiliza el contexto recuperado (`retrieved_knowledge`) y la respuesta ya generada (`generated_answer`) almacenados en el checkpoint.
- No vuelve a consultar BM25, embeddings semánticos, Reranker ni el modelo SLM local.
- Sincroniza las correcciones de inmediato con Google Drive.

In [ ]:
# =====================================================================
# 🔄 REEVALUACIÓN SELECTIVA DE LLM JUDGE EN SISTEMA RAG
# =====================================================================
FIX_PREVIOUS_ZEROS = True  # True para reparar muestras previas guardadas con 0.0

print(
    f"[INFO] Verificando '{CHECKPOINT_FILE}' en busca de llamadas pendientes...")
records_to_check = []
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    records_to_check.append(json.loads(line.strip()))
                except Exception:
                    pass

pending_triad = []
pending_geval = []

for i, r in enumerate(records_to_check):
    m = r.get("metrics", {})
    gen_ans = r.get("generated_answer", "")
    if not gen_ans.strip():
        continue

    # Tríada RAG
    triad_failed = (
        m.get("judge_triad_status") == "FAILED_API" or
        m.get("faithfulness") is None or
        m.get("answer_relevancy") is None or
        (FIX_PREVIOUS_ZEROS and (m.get("faithfulness")
         == 0.0 and m.get("answer_relevancy") == 0.0))
    )
    if triad_failed:
        pending_triad.append(i)

    # G-Eval
    geval_failed = (
        m.get("judge_geval_status") == "FAILED_API" or
        m.get("geval_coherence") is None or
        m.get("geval_consistency") is None or
        m.get("geval_fluency") is None or
        (FIX_PREVIOUS_ZEROS and (m.get("geval_coherence") == 0.0 or m.get(
            "geval_consistency") == 0.0 or m.get("geval_fluency") == 0.0))
    )
    if geval_failed:
        pending_geval.append(i)

all_pending = sorted(list(set(pending_triad + pending_geval)))
print(
    f"📊 Muestras pendientes de Tríada: {len(pending_triad)} | G-Eval: {len(pending_geval)} | Total únicas: {len(all_pending)}/{len(records_to_check)}")

if not all_pending:
    print("✨ ¡Todas las métricas de LLM Judge en el sistema RAG están completas y válidas!")
else:
    print("\n🚀 Reevaluando llamadas pendientes (sin re-ejecutar RAG ni SLM)...")
    repaired_triad = 0
    repaired_geval = 0
    for idx in tqdm(all_pending, desc="Reparando LLM Judge RAG"):
        rec = records_to_check[idx]
        m = rec.setdefault("metrics", {})
        q = rec.get("question", "")
        k = rec.get("retrieved_knowledge", "")
        ans = rec.get("generated_answer", "")

        if idx in pending_triad:
            res_t = eval_rag_triad(q, k, ans, max_retries=5)
            if res_t and res_t.get("status") == "SUCCESS":
                m["context_relevancy"] = res_t["context_relevancy"]
                m["faithfulness"] = res_t["faithfulness"]
                m["answer_relevancy"] = res_t["answer_relevancy"]
                m["triad_reasoning"] = res_t["reasoning"]
                m["judge_triad_status"] = "SUCCESS"
                repaired_triad += 1
            else:
                m["context_relevancy"] = None
                m["faithfulness"] = None
                m["answer_relevancy"] = None
                m["judge_triad_status"] = "FAILED_API"

        if idx in pending_geval:
            res_g = eval_geval(q, ans, max_retries=5)
            if res_g and res_g.get("status") == "SUCCESS":
                m["geval_coherence"] = res_g["coherence"]
                m["geval_consistency"] = res_g["consistency"]
                m["geval_fluency"] = res_g["fluency"]
                m["judge_geval_status"] = "SUCCESS"
                repaired_geval += 1
            else:
                m["geval_coherence"] = None
                m["geval_consistency"] = None
                m["geval_fluency"] = None
                m["judge_geval_status"] = "FAILED_API"

        time.sleep(0.5)

    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        for r in records_to_check:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    if drive_dest_path:
        try:
            shutil.copy(CHECKPOINT_FILE, drive_dest_path)
            print(
                f"☁️ Checkpoint RAG actualizado y respaldado en Google Drive: {drive_dest_path}")
        except Exception as e:
            print(f"ℹ️ Archivo local '{CHECKPOINT_FILE}' actualizado ({e}).")

    results_list = records_to_check
    print(
        f"\n✅ ¡Reparación finalizada! Tríadas corregidas: {repaired_triad}, G-Eval corregidas: {repaired_geval}.")

## 9. Reporte Ejecutivo, Estadísticas y Dashboard Visual del Sistema RAG


### (Opcional) Cargar archivo de resultados previo
Si reiniciaste el entorno y ya tienes un archivo de resultados `.jsonl`, puedes subirlo en la siguiente celda para generar el reporte sin volver a ejecutar la evaluación.


In [ ]:
import json
import os

print("📂 Si deseas visualizar un archivo de resultados previo, súbelo aquí:")
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        upload_name = list(uploaded.keys())[0]
        results_list = []
        with open(upload_name, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    results_list.append(json.loads(line.strip()))
        print(
            f"✅ ¡Cargadas {len(results_list)} muestras desde '{upload_name}'!")
    else:
        print(
            f"ℹ️ No se subió archivo. Usando resultados actuales en memoria ({len(results_list)} muestras).")
except Exception as e:
    print(f"ℹ️ Usando resultados en memoria ({len(results_list)} muestras).")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from IPython.display import display

# 1. Preparar DataFrame
df_res = pd.json_normalize([r["metrics"] for r in results_list])
n_samples = len(df_res)

# 2. Definición de Métricas y Umbrales
# Formato: (Nombre legible, columna, factor_escala, [umbral_excelente, umbral_bueno, umbral_regular], invertido)
METRIC_DEFS = [
    # Tríada RAG (0 a 5)
    ("1. Context Relevancy (RAG Triad)",
     "context_relevancy",   1, [4.5, 3.5, 2.5], False),
    ("2. Faithfulness (RAG Triad)",
     "faithfulness",        1, [4.5, 3.5, 2.5], False),
    ("3. Answer Relevancy (RAG Triad)",
     "answer_relevancy",    1, [4.5, 3.5, 2.5], False),

    # G-Eval Calidad Lingüística (1 a 5)
    ("4. G-Eval Coherencia",
     "geval_coherence",     1, [4.5, 3.5, 2.5], False),
    ("   G-Eval Consistencia",
     "geval_consistency",   1, [4.5, 3.5, 2.5], False),
    ("   G-Eval Fluidez",                 "geval_fluency",
     1, [4.5, 3.5, 2.5], False),

    # BERTScore vs Referencia (0 a 1)
    ("5. BERTScore Precision",
     "bertscore_precision", 1, [0.85, 0.70, 0.50], False),
    ("   BERTScore Recall",               "bertscore_recall",
     1, [0.85, 0.70, 0.50], False),
    ("   BERTScore F1 Score",             "bertscore_f1",
     1, [0.85, 0.70, 0.50], False),

    # Latencias (segundos - menor es mejor)
    ("6. Latencia Recuperación RAG (s)",
     "time_retrieval",      1, [0.30, 0.80, 1.50], True),
    ("   Latencia Generación SLM (s)",
     "time_generation",     1, [2.00, 4.00, 7.00], True),
    ("   Latencia Total E2E (s)",         "time_e2e",
     1, [2.50, 5.00, 8.50], True),

    # Rendimiento
    ("7. Velocidad Generación (tok/s)",
     "tokens_per_second",   1, [30.0, 20.0, 10.0], False),
]

TABLE_COLORS = {
    "Excelente": "#c6efce",
    "Bueno":     "#ffeb9c",
    "Regular":   "#ffd8b1",
    "Bajo":      "#ffc7ce"
}


def calificar(col, valor, umbrales, invertido):
    a, b, c = umbrales
    if invertido:
        if valor <= a:
            return "Excelente"
        if valor <= b:
            return "Bueno"
        if valor <= c:
            return "Regular"
        return "Bajo"
    else:
        if valor >= a:
            return "Excelente"
        if valor >= b:
            return "Bueno"
        if valor >= c:
            return "Regular"
        return "Bajo"


rows = []
for nombre, col, factor, umbrales, invertido in METRIC_DEFS:
    if col in df_res.columns:
        promedio = df_res[col].mean() * factor
        std = df_res[col].std() * factor
    else:
        promedio, std = 0.0, 0.0
    rows.append({
        "Métrica": nombre,
        "Promedio": promedio,
        "Desv. Estándar": std,
        "Evaluación": calificar(col, promedio, umbrales, invertido)
    })

summary_stats = pd.DataFrame(rows)


def resaltar_fila(row):
    color = TABLE_COLORS.get(row["Evaluación"], "#ffffff")
    return [f"background-color: {color}; color: #1a1a1a"] * len(row)


def emoji_evaluacion(v):
    return {"Excelente": "🟢 Excelente", "Bueno": "🟡 Bueno", "Regular": "🟠 Regular", "Bajo": "🔴 Bajo"}.get(v, v)


summary_display = summary_stats.copy()
summary_display["Evaluación"] = summary_display["Evaluación"].map(
    emoji_evaluacion)

styled = (
    summary_display.style
    .apply(lambda row: resaltar_fila(summary_stats.loc[row.name]), axis=1)
    .format({"Promedio": "{:,.3f}", "Desv. Estándar": "{:,.3f}"})
    .set_properties(**{"text-align": "left", "font-size": "13px", "color": "#1a1a1a"})
    .set_table_styles([{"selector": "th", "props": [
        ("text-align", "left"), ("background-color", "#1b4d3e"),
        ("color", "white"), ("font-size", "13px")
    ]}])
    .hide(axis="index")
)

print("=" * 90)
print(
    f" 📊 REPORTE DE EVALUACIÓN END-TO-END — SISTEMA RAG FARMIFAI | Muestras: {n_samples}")
print("=" * 90)
display(styled)
print("Leyenda: 🟢 Excelente   🟡 Bueno   🟠 Regular   🔴 Bajo\n")

In [ ]:
plt.rcParams.update({"font.size": 10})
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.25,
                      top=0.92, bottom=0.06, left=0.06, right=0.96)

fig.suptitle("🚜 DASHBOARD DE RENDIMIENTO Y CALIDAD DEL SISTEMA RAG FARMIFAI",
             fontsize=18, fontweight="bold", y=0.97)

ax_radar = fig.add_subplot(gs[0, 0], projection="polar")
ax_latency = fig.add_subplot(gs[0, 1])
ax_speed = fig.add_subplot(gs[1, 0])
ax_corr = fig.add_subplot(gs[1, 1])

# --- 1. GRÁFICA DE RADAR: CALIDAD HOLÍSTICA NORMALIZADA (0 - 100%) ---
radar_labels = [
    "Context\nRelevancy", "Faithfulness\n(Groundedness)", "Answer\nRelevancy",
    "G-Eval\nCoherencia", "G-Eval\nFluidez", "BERTScore\nF1"
]
radar_values = [
    (df_res["context_relevancy"].mean() / 5.0) *
    100 if "context_relevancy" in df_res else 0.0,
    (df_res["faithfulness"].mean() / 5.0) *
    100 if "faithfulness" in df_res else 0.0,
    (df_res["answer_relevancy"].mean() / 5.0) *
    100 if "answer_relevancy" in df_res else 0.0,
    (df_res["geval_coherence"].mean() / 5.0) *
    100 if "geval_coherence" in df_res else 0.0,
    (df_res["geval_fluency"].mean() / 5.0) *
    100 if "geval_fluency" in df_res else 0.0,
    (df_res["bertscore_f1"].mean() * 100) if "bertscore_f1" in df_res else 0.0,
]

num_vars = len(radar_labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]
radar_plot_values = radar_values + radar_values[:1]

ax_radar.plot(angles, radar_plot_values, color="#1b4d3e",
              linewidth=2.5, linestyle="solid")
ax_radar.fill(angles, radar_plot_values, color="#2e7d32", alpha=0.30)
ax_radar.set_yticklabels(
    ["20%", "40%", "60%", "80%", "100%"], color="#666666", size=8)
ax_radar.set_ylim(0, 100)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_labels, fontweight="bold", size=10)
ax_radar.set_title("A. Perfil Holístico de Calidad RAG (%)",
                   size=13, fontweight="bold", pad=20)

for angle, val in zip(angles[:-1], radar_values):
    ax_radar.text(angle, val + 6, f"{val:.1f}%", ha="center",
                  va="center", size=9, fontweight="bold", color="#1b4d3e")

# --- 2. DESGLOSE DE LATENCIA POR COMPONENTE ---
lat_components = ["BM25", "Semántica (E5)", "Reranker", "Generación SLM"]
mean_times = [
    df_res["time_bm25"].mean() if "time_bm25" in df_res else 0.0,
    df_res["time_semantic"].mean() if "time_semantic" in df_res else 0.0,
    df_res["time_rerank"].mean() if "time_rerank" in df_res else 0.0,
    df_res["time_generation"].mean() if "time_generation" in df_res else 0.0,
]
colors_lat = ["#4a90e2", "#50e3c2", "#f5a623", "#d0021b"]

y_pos = np.arange(len(lat_components))
bars = ax_latency.barh(y_pos, mean_times, color=colors_lat,
                       edgecolor="black", alpha=0.85)
ax_latency.set_yticks(y_pos)
ax_latency.set_yticklabels(lat_components, fontweight="bold")
ax_latency.set_xlabel("Tiempo Promedio (segundos)", fontweight="bold")
ax_latency.set_title(
    f"B. Desglose de Latencia del Pipeline (Total E2E: {df_res['time_e2e'].mean():.2f}s)", size=13, fontweight="bold")
ax_latency.grid(axis="x", linestyle="--", alpha=0.5)

for bar in bars:
    w = bar.get_width()
    ax_latency.text(w + 0.02, bar.get_y() + bar.get_height()/2,
                    f"{w:.3f} s", va="center", size=9, fontweight="bold")

# --- 3. VELOCIDAD DE GENERACIÓN (TOKENS / SEGUNDO) ---
tok_s = df_res["tokens_per_second"] if "tokens_per_second" in df_res else pd.Series([
                                                                                    0])
ax_speed.hist(tok_s, bins=15, color="#1b4d3e",
              edgecolor="white", alpha=0.75, density=False)
mean_tok = tok_s.mean()
median_tok = tok_s.median()
ax_speed.axvline(mean_tok, color="#d0021b", linestyle="--",
                 linewidth=2, label=f"Media: {mean_tok:.1f} tok/s")
ax_speed.axvline(median_tok, color="#f5a623", linestyle="-.",
                 linewidth=2, label=f"Mediana: {median_tok:.1f} tok/s")
ax_speed.set_title(
    "C. Distribución de Throughput del SLM (tokens/s)", size=13, fontweight="bold")
ax_speed.set_xlabel("Tokens por Segundo", fontweight="bold")
ax_speed.set_ylabel("Frecuencia (Consultas)", fontweight="bold")
ax_speed.legend(loc="upper right")
ax_speed.grid(axis="y", linestyle="--", alpha=0.5)

# --- 4. CORRELACIÓN: CONTEXT RELEVANCY VS FAITHFULNESS ---
c_rel = df_res["context_relevancy"] if "context_relevancy" in df_res else pd.Series([
                                                                                    0])
faith = df_res["faithfulness"] if "faithfulness" in df_res else pd.Series([0])

scatter = ax_corr.scatter(
    c_rel + np.random.uniform(-0.08, 0.08, size=len(c_rel)),
    faith + np.random.uniform(-0.08, 0.08, size=len(faith)),
    c=df_res["bertscore_f1"] if "bertscore_f1" in df_res else "#1b4d3e",
    cmap="Greens",
    edgecolor="#1b4d3e",
    alpha=0.75,
    s=65
)
cbar = plt.colorbar(scatter, ax=ax_corr)
cbar.set_label("BERTScore F1", fontweight="bold")

# Línea de tendencia
if len(c_rel) > 1 and c_rel.std() > 0:
    m, b = np.polyfit(c_rel, faith, 1)
    x_line = np.linspace(0, 5, 100)
    ax_corr.plot(x_line, m * x_line + b, color="#d0021b", linewidth=2,
                 linestyle="--", label=f"Tendencia (Pendiente: {m:.2f})")
    ax_corr.legend(loc="upper left")

ax_corr.set_xlim(-0.5, 5.5)
ax_corr.set_ylim(-0.5, 5.5)
ax_corr.set_xlabel(
    "Relevancia del Contexto Recuperado (0 a 5)", fontweight="bold")
ax_corr.set_ylabel(
    "Fidelidad de la Respuesta / Faithfulness (0 a 5)", fontweight="bold")
ax_corr.set_title(
    "D. Impacto del Retrieval en la Fidelidad del SLM", size=13, fontweight="bold")
ax_corr.grid(True, linestyle="--", alpha=0.5)

plt.show()

In [ ]:
print("=" * 80)
print(" 🔬 ANÁLISIS CUALITATIVO DEL SISTEMA RAG: MEJORES Y PEORES RESPUESTAS")
print("=" * 80)

# Score global compuesto del sistema RAG (0 a 100)
df_res["global_score"] = (
    ((df_res["context_relevancy"] / 5.0) * 20) +
    ((df_res["faithfulness"] / 5.0) * 35) +
    ((df_res["answer_relevancy"] / 5.0) * 25) +
    (df_res["bertscore_f1"] * 20)
)

best_idx = df_res["global_score"].idxmax()
worst_idx = df_res["global_score"].idxmin()

best_case = results_list[best_idx]
worst_case = results_list[worst_idx]


def print_case_card(case, title, emoji_icon):
    idx = case["sample_index"]
    score = df_res.loc[df_res["sample_index"] == idx,
                       "global_score"].values[0] if "sample_index" in df_res.columns else df_res.loc[idx, "global_score"]
    m = case["metrics"]

    print(
        f"\n{emoji_icon} {title} [Muestra #{idx}] — Score Global: {score:.1f}/100")
    print(f"❓ PREGUNTA: {case['question']}")
    print(
        f"\n📦 CHUNKS RECUPERADOS POR EL RAG (Top-{len(case['retrieved_chunks'])}):")
    for i, c in enumerate(case["retrieved_chunks"], 1):
        snippet = c.get("text", "").replace("\n", " ")[:140]
        print(f"   [{i}] Doc: {c.get('document_id')} (Fragmento {c.get('chunk_number')}) | Score Rerank: {c.get('rerank_score', 0.0):.4f}")
        print(f"       \"{snippet}...\"")
    print(f"\n💬 RESPUESTA GENERADA POR EL SLM:\n{case['generated_answer']}")
    print(f"\n🎯 RESPUESTA DE REFERENCIA:\n{case['reference_answer']}")
    print(f"\n📊 DIAGNÓSTICO:")
    print(
        f"   • Tríada RAG : Context Relevancy={m['context_relevancy']}/5 | Faithfulness={m['faithfulness']}/5 | Answer Relevancy={m['answer_relevancy']}/5")
    print(f"   • Justificación del Juez : {m.get('triad_reasoning', 'N/A')}")
    print(
        f"   • BERTScore  : F1={m['bertscore_f1']:.3f} (P={m['bertscore_precision']:.3f}, R={m['bertscore_recall']:.3f})")
    print(
        f"   • Latencias  : Recuperación={m['time_retrieval']:.3f}s | Generación={m['time_generation']:.2f}s | Total E2E={m['time_e2e']:.2f}s | Vel={m['tokens_per_second']:.1f} tok/s")
    print("-" * 80)


print_case_card(
    best_case, "MEJOR CASO DEL SISTEMA RAG (Alta Relevancia y Fidelidad)", "🏆")
print_case_card(worst_case, "CASO DESAFIANTE / OPORTUNIDAD DE MEJORA", "⚠️")